[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/expand_dataset.ipynb)

# Expand a Dataset

Take 500 rows of [GSM8K](https://huggingface.co/datasets/openai/gsm8k) from Hugging Face, then grow them three ways - **domain**, **general** and **similar** rows, 100 of each and inspect what each kind of row looks like.

**Docs:** [/adaptive-data/expand-data/](https://docs.adaptionlabs.ai/adaptive-data/expand-data/) · [runbook page](https://docs.adaptionlabs.ai/runbooks/expand-a-dataset/)  
**Blog:** [Expand: Automatic Data Coverage for Custom Model Training](https://adaptionlabs.ai/blog/expand)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

## Overview

**Expand** takes you from a handful of examples of the capability you care about to a large-scale training dataset — start with as few as 10 examples and grow to many thousands of data points similar to your tasks and optimized for AI training. Models trained on too few examples often fail on phrasing and situations they haven't seen; Expand generates the additional examples a model needs to respond to that variation.

It grows a dataset three ways:

- **Similar rows** — synthetic examples in the style of your own data, matching its topics, language, and format.
- **Domain rows** — more tasks from the same domain, so fewer tasks are unfamiliar to your model.
- **General rows** — other domains for balance, since training quality improves when the data is varied.

This runbook demonstrates all three on a GSM8K seed. Background: [Expand: Automatic Data Coverage for Custom Model Training](https://adaptionlabs.ai/blog/expand).

In [ ]:
%pip install -q adaption pandas pyarrow

## Setup

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

In [ ]:
PARQUET_URL = "https://huggingface.co/datasets/openai/gsm8k/resolve/main/main/train-00000-of-00001.parquet"
NUM_ROWS = 500  # rows of GSM8K to upload; this is the dataset we expand

# One expansion per strategy, so each output dataset holds the seed plus one kind of row.
EXPANSIONS = {
    "domain": {"domain_rows": 100},
    "general": {"general_rows": 100},
    "similar": {"similar_rows": 100},
}

SAMPLES_TO_SHOW = 3

## Helper functions

A few small utilities used below:

- `wait` - blocks until a dataset job finishes and raises with the API's error message if it failed.
- `download` - downloads a dataset as JSONL and loads it into a DataFrame.
- `added_rows` - picks out the rows an expansion added (prompt not among the seed prompts).

In [ ]:
import pandas as pd
from IPython.display import display


def wait(dataset_id: str, what: str):
    """Wait for a dataset job and raise with the API's message if it failed."""
    status = client.datasets.wait_for_completion(dataset_id, timeout=3 * 3600)
    if status.status == "failed":
        msg = (status.error_data and status.error_data.message) or "unknown error"
        raise RuntimeError(f"{what} failed: {msg}")
    print(f"{what}: {status.status}, {status.row_count} rows")
    return status


def download(dataset_id: str, path: str) -> pd.DataFrame:
    client.datasets.download(dataset_id, file_format="jsonl").write_to_file(path)
    return pd.read_json(path, lines=True)


PROMPT_COLS = ["original_prompt", "enhanced_prompt", "prompt"]
COMPLETION_COLS = ["original_completion", "enhanced_completion", "completion"]


def first_present(row, columns):
    for col in columns:
        value = row.get(col)
        if isinstance(value, str) and value.strip() and value.strip() != "N/A":
            return value
    return ""


def added_rows(df: pd.DataFrame, seed_prompts: set) -> pd.DataFrame:
    """Rows the expansion added - those whose prompt is not among the seed prompts."""
    prompts = df.apply(lambda r: first_present(r, PROMPT_COLS), axis=1)
    return df[~prompts.isin(seed_prompts)]

## Step 1 - Take a 500-row slice of GSM8K

GSM8K is ~7.5k grade-school math word problems, each with a step-by-step solution. We download the train split from Hugging Face and keep a random 500 rows.

In [ ]:
import urllib.request

urllib.request.urlretrieve(PARQUET_URL, "gsm8k-train.parquet")
seed_df = pd.read_parquet("gsm8k-train.parquet").sample(NUM_ROWS, random_state=0)
seed_df.to_parquet("gsm8k-seed.parquet", index=False)
print(f"{len(seed_df)} rows, columns: {list(seed_df.columns)}")

## Step 2 - Upload

Expansion needs a dataset that is ready to read. Instead of running Adaptive Data on it, we upload the rows with `processing_mode="raw"`, which takes them as-is: `question` becomes the prompt and `answer` the completion. See [`datasets.upload_file`](https://docs.adaptionlabs.ai/api/python/resources/datasets/methods/create).

In [ ]:
upload = client.datasets.upload_file(
    "gsm8k-seed.parquet",
    name="GSM8K seed (500 rows)",
    processing_mode="raw",
    column_mapping={"prompt": "question", "completion": "answer"},
)
dataset_id = upload.dataset_id
print(dataset_id)

wait(dataset_id, "Upload")

In [ ]:
seed_view = seed_df.rename(columns={"question": "prompt", "answer": "completion"})
seed_prompts = set(seed_view["prompt"])
display(seed_view[["prompt", "completion"]].head(SAMPLES_TO_SHOW))

## Step 3 - Expand: domain, general and similar rows

[`datasets.augment`](https://docs.adaptionlabs.ai/api/python/resources/datasets/methods/augment) returns a **new dataset** holding the seed rows plus the added ones; the seed is left unchanged. It supports three kinds of row:

| Strategy | Parameter | Where rows come from |
| --- | --- | --- |
| Domain | `domain_rows` | Rows **matching the topics** of your dataset |
| General | `general_rows` | Rows from **other topics**, to broaden coverage |
| Similar | `similar_rows` | Rows in the style, language and format of the samples you provided |

The three can be combined in a single call. Here we launch one expansion per strategy so each output contains just one kind of added row and we can compare them side by side. First, price all three:

In [ ]:
total = 0
for name, rows in EXPANSIONS.items():
    est = client.datasets.augment(dataset_id, **rows, estimate=True)
    total += est.estimated_credits_consumed
    print(f"{name:>8}: {est.estimated_credits_consumed} credits")
print(f"{'total':>8}: {total} credits")

Launch all three. The `idempotency_key` makes re-running this cell safe: a repeat returns the dataset already created instead of starting (and charging for) a second run.

In [ ]:
expanded_ids = {}
for name, rows in EXPANSIONS.items():
    response = client.datasets.augment(
        dataset_id,
        **rows,
        idempotency_key=f"expand-runbook-{dataset_id}-{name}",
    )
    expanded_ids[name] = response.dataset_id
    print(f"{name:>8}: {response.dataset_id} ({response.status})")

Wait for each expansion. A timeout only stops waiting, not the job — re-run the cell to resume.

In [ ]:
for name, expanded_id in expanded_ids.items():
    wait(expanded_id, f"{name} expansion")

## Step 4 - Inspect what each strategy added

Each expansion carries the seed rows plus the rows it added. The download doesn't tag origin, so `added_rows` treats any row whose prompt isn't in the seed as added. Because each dataset ran a single strategy, its added rows all come from that strategy.

In [ ]:
for name, expanded_id in expanded_ids.items():
    df = download(expanded_id, f"gsm8k-{name}.jsonl")
    added = added_rows(df, seed_prompts)
    print(f"{name}: {len(df)} rows = {len(df) - len(added)} seed + {len(added)} added")
    view = pd.DataFrame(
        {
            "prompt": added.apply(lambda r: first_present(r, PROMPT_COLS), axis=1),
            "completion": added.apply(lambda r: first_present(r, COMPLETION_COLS), axis=1),
        }
    )
    display(view.head(SAMPLES_TO_SHOW))


## Next steps

- Combine the seed and the expansions you want into one training set with [`datasets.combine`](https://docs.adaptionlabs.ai/adaptive-data/combine-datasets/), or request all three kinds in one `augment` call.
- Add other languages with [`translate` and `localize`](https://docs.adaptionlabs.ai/adaptive-data/expand-data/#translate-rows).
- Train on the result with the [AutoScientist runbook](../autoscientist/autoscientist.ipynb).